# Module 9 Worksheet — Caching, Streaming, Rate Limiting
**Corrected in this version:** `cached_call()` and the decision-quiz loop now use `ask()` instead of `multimodal_chat()`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. A simple cache, and measuring the savings

In [ ]:
import hashlib, time

cache = {}

def cached_call(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    key = hashlib.sha256(f"{system_prompt}|{user_prompt}|{model}|{max_tokens}".encode()).hexdigest()
    if key in cache:
        return cache[key], True  # True = cache hit
    result = ask(system_prompt, user_prompt, model=model, max_tokens=max_tokens)
    cache[key] = result
    return result, False

t0 = time.time()
r1, hit1 = cached_call("Be concise.", "What is RAG?")
t1 = time.time()
r2, hit2 = cached_call("Be concise.", "What is RAG?")  # identical request
t2 = time.time()

print(f"First call: hit={hit1}, latency={t1-t0:.2f}s")
print(f"Second call (identical): hit={hit2}, latency={t2-t1:.2f}s")

## 2. Simulated streaming vs non-streaming (perceived latency, this module's teaser)

In [ ]:
import time

def non_streaming_demo(text, delay_per_char=0.02):
    t0 = time.time()
    time.sleep(len(text) * delay_per_char)
    print(text)
    print(f"[non-streaming: nothing appeared until {time.time()-t0:.1f}s in]")

def streaming_demo(text, delay_per_char=0.02):
    t0 = time.time()
    for ch in text:
        print(ch, end="", flush=True)
        time.sleep(delay_per_char)
    print(f"\n[streaming: first character appeared almost immediately, total time same: {time.time()-t0:.1f}s]")

demo_text = "RAG combines retrieval and generation to ground answers in real data."
print("--- Non-streaming ---")
non_streaming_demo(demo_text)
print("\n--- Streaming ---")
streaming_demo(demo_text)

## 3. A toy rate limiter

In [ ]:
import time
from collections import deque

class RateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window = window_seconds
        self.timestamps = deque()

    def allow(self):
        now = time.time()
        while self.timestamps and now - self.timestamps[0] > self.window:
            self.timestamps.popleft()
        if len(self.timestamps) < self.max_requests:
            self.timestamps.append(now)
            return True
        return False

limiter = RateLimiter(max_requests=3, window_seconds=5)
for i in range(6):
    print(f"Request {i+1}: {'allowed' if limiter.allow() else 'RATE LIMITED'}")

## 4. Fine-tune vs RAG vs Prompt Engineering — decision quiz

In [ ]:
scenarios = [
    "Your support bot needs to always respond in a very specific brand voice/tone.",
    "Your support bot needs to answer questions about a product catalog that changes weekly.",
    "Your model needs to reliably output a deeply specific internal data format that prompting keeps getting wrong, and you have 5,000 labeled examples.",
]
for s in scenarios:
    answer = ask(
        "Given the scenario, recommend ONE of: prompt engineering, RAG, fine-tuning. One sentence justification.",
        s, max_tokens=100,
    )
    print(f"Scenario: {s}\n-> {answer}\n")

## Teaser exercise
Combine all three: add caching to the RAG pipeline from Module 3, add a rate limiter in front of it, and confirm cached requests bypass the rate limiter entirely (since they never reach the model) — that's the actual production benefit of caching, not just speed.